In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean visual style for our upcoming charts
sns.set_theme(style="whitegrid")

# 1. Load the data 
# We use low_memory=False because the dataset has many columns with mixed data types
file_path = '../data/loan.csv'  # Update 'loan.csv' if your file is named differently
df = pd.read_csv(file_path, low_memory=False)

# 2. Basic Inspection
print("--- Data Loading Complete ---")
print(f"Dataset Shape (Rows, Columns): {df.shape}\n")

# 3. View the first 5 rows to see what the raw data looks like
df.head()

--- Data Loading Complete ---
Dataset Shape (Rows, Columns): (2260668, 145)



,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,NaN,NaN,2500,2500,2500.0,36 months,13.56,84.92,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,30000,30000,30000.0,60 months,18.94,777.23,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,5000,5000,5000.0,36 months,17.97,180.69,D,D1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,4000,4000,4000.0,36 months,18.94,146.51,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,30000,30000,30000.0,60 months,16.14,731.78,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
!pip install pyarrow


   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
    --------------------------------------- 0.5/27.5 MB 6.9 MB/s eta 0:00:04
   -- ------------------------------------- 1.6/27.5 MB 5.1 MB/s eta 0:00:06
   --- ------------------------------------ 2.6/27.5 MB 5.2 MB/s eta 0:00:05
   ----- ---------------------------------- 3.9/27.5 MB 5.6 MB/s eta 0:00:05
   ------ --------------------------------- 4.7/27.5 MB 5.4 MB/s eta 0:00:05
   -------- ------------------------------- 6.0/27.5 MB 5.5 MB/s eta 0:00:04
   ----------- ---------------------------- 7.6/27.5 MB 5.7 MB/s eta 0:00:04
   ------------ --------------------------- 8.9/27.5 MB 6.0 MB/s eta 0:00:04
   -------------- ------------------------- 9.7/27.5 MB 5.5 MB/s eta 0:00:04
   --------------- ------------------------ 10.5/27.5 MB 5.4 MB/s eta 0:00:04
   ---------------- ----------------------- 11.3/27.5 MB 5.1 MB/s eta 0:00:04
   ----------------- ---------------------- 12.1/27.5 MB 5.1 MB/s eta 0:00:04
   


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
!python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------------------------- ---------- 1.3/1.8 MB 5.4 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 5.2 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2


In [3]:
# 1. Force all column headers to be explicitly strings
df.columns = df.columns.astype(str)

# 2. Drop any accidentally duplicated columns (just in case!)
df = df.loc[:, ~df.columns.duplicated()]

# 3. Save to parquet, and tell it to ignore the index to prevent schema errors
df.to_parquet('../data/loan.parquet', engine='pyarrow', index=False)

print("Saved as Parquet successfully!")

Saved as Parquet successfully!


In [4]:
print("Cleaning up messy column types...")

# 1. Force all column headers to be strings
df.columns = df.columns.astype(str)

# 2. Drop duplicated columns
df = df.loc[:, ~df.columns.duplicated()]

# 3. Find all columns that have mixed or 'object' data types and force them to be strings
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str)

# 4. Try saving to Parquet, with a foolproof Pickle backup
try:
    df.to_parquet('../data/loan.parquet', engine='pyarrow', index=False)
    print("Success! Saved as Parquet. 🚀")
except Exception as e:
    print("Parquet is still being stubborn. Using Pickle format instead...")
    df.to_pickle('../data/loan.pkl')
    print("Success! Saved as Pickle. 🚀")

Cleaning up messy column types...
Success! Saved as Parquet. 🚀


In [5]:
# Calculate the percentage of missing values for each column
missing_data = df.isnull().mean() * 100

# Filter to see only columns that have more than 0% missing values
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

# Display the top 20 worst offenders
print("\nTop 20 columns with the most missing data (Percentage %):")
print(missing_data.head(20))


Top 20 columns with the most missing data (Percentage %):
id                                            100.000000
member_id                                     100.000000
url                                           100.000000
orig_projected_additional_accrued_interest     99.627278
hardship_payoff_balance_amount                 99.530537
deferral_term                                  99.530537
hardship_dpd                                   99.530537
hardship_amount                                99.530537
hardship_last_payment_amount                   99.530537
hardship_length                                99.530537
settlement_amount                              98.537777
settlement_percentage                          98.537777
settlement_term                                98.537777
sec_app_mths_since_last_major_derog            98.410116
sec_app_revol_util                             95.302981
revol_bal_joint                                95.221766
sec_app_open_acc             

In [6]:
print(f"Original shape: {df.shape}")

# 1. Define our threshold (50% missing)
threshold = 50 

# 2. Find the columns that exceed this threshold
columns_to_drop = missing_data[missing_data > threshold].index

# 3. Drop those columns from our dataframe
df = df.drop(columns=columns_to_drop)

print(f"Dropped {len(columns_to_drop)} columns because they were mostly empty.")
print(f"New Dataset Shape: {df.shape}")

Original shape: (2260668, 145)
Dropped 30 columns because they were mostly empty.
New Dataset Shape: (2260668, 115)


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Loading data...")
# 1. Load the fast Pickle file we made yesterday
df = pd.read_pickle('../data/loan.pkl')

# 2. Re-apply the 50% missing data rule
missing_data = df.isnull().mean() * 100
columns_to_drop = missing_data[missing_data > 50].index
df = df.drop(columns=columns_to_drop)

print(f"Data ready! Current Shape: {df.shape}")
print("-" * 30)

# 3. Check the target variable categories
print("Breakdown of Loan Statuses:")
print(df['loan_status'].value_counts(dropna=False))

Loading data...


NotImplementedError: (<StringDtype(storage='python', na_value=nan)>, array([' 36 months', ' 60 months', ' 36 months', ..., ' 36 months',
       ' 60 months', ' 60 months'], shape=(2260668,), dtype=object))

In [10]:
# Check the unique values in our target variable and their counts
print("Breakdown of Loan Statuses:")
print(df['loan_status'].value_counts(dropna=False))

# Let's also see this as a percentage to understand our class imbalance
print("\nPercentage Breakdown:")
print(df['loan_status'].value_counts(normalize=True) * 100)

Breakdown of Loan Statuses:
loan_status
Fully Paid                                             1041952
Current                                                 919695
Charged Off                                             261655
Late (31-120 days)                                       21897
In Grace Period                                           8952
Late (16-30 days)                                         3737
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     31
Name: count, dtype: int64

Percentage Breakdown:
loan_status
Fully Paid                                             46.090448
Current                                                40.682444
Charged Off                                            11.574234
Late (31-120 days)                                      0.968608
In Grace Period                                         0.395989
Late (1

In [11]:
print(f"Rows before filtering: {df.shape[0]}")

# 1. Define our specific categories
bad_statuses = ['Charged Off', 'Default', 'Late (31-120 days)']
good_statuses = ['Fully Paid']

# 2. Filter the dataframe to keep ONLY these defined statuses
df = df[df['loan_status'].isin(good_statuses + bad_statuses)].copy()

# 3. Create the new binary target column (1 if in bad_statuses, 0 otherwise)
df['target'] = np.where(df['loan_status'].isin(bad_statuses), 1, 0)

# 4. Drop the old text column as we no longer need it
df = df.drop(columns=['loan_status'])

print(f"Rows after filtering out incomplete loans: {df.shape[0]}")
print("-" * 30)
print("New Target Class Imbalance (0 = Good, 1 = Bad):")
print(df['target'].value_counts(normalize=True) * 100)

Rows before filtering: 2260668
Rows after filtering out incomplete loans: 1325535
------------------------------
New Target Class Imbalance (0 = Good, 1 = Bad):
target
0    78.606148
1    21.393852
Name: proportion, dtype: float64


In [12]:
# 1. Isolate only the numerical columns for our math
numeric_cols = df.select_dtypes(include=[np.number]).columns

# 2. Calculate how each numerical feature correlates with our 'target' (1 = Default)
print("Crunching correlations... this might take 10-20 seconds...\n")
correlations = df[numeric_cols].corr()['target'].sort_values()

# 3. Display the features most strongly linked to Defaulting
print("🔴 Top 10 Features POSITIVELY correlated with Default (Higher value = Higher risk):")
print(correlations.tail(11).head(10)) # We tail 11 and head 10 to exclude 'target' correlating with itself

print("\n" + "-"*40 + "\n")

# 4. Display the features most strongly linked to Paying off the loan
print("🟢 Top 10 Features NEGATIVELY correlated with Default (Higher value = Lower risk):")
print(correlations.head(10))

Crunching correlations... this might take 10-20 seconds...

🔴 Top 10 Features POSITIVELY correlated with Default (Higher value = Higher risk):
dti                        0.087715
all_util                   0.089277
acc_open_past_24mths       0.100074
total_rec_late_fee         0.155128
out_prncp_inv              0.198378
out_prncp                  0.198389
int_rate                   0.263410
collection_recovery_fee    0.450344
recoveries                 0.475074
target                     1.000000
Name: target, dtype: float64

----------------------------------------

🟢 Top 10 Features NEGATIVELY correlated with Default (Higher value = Lower risk):
total_rec_prncp   -0.448112
last_pymnt_amnt   -0.360365
total_pymnt       -0.324732
total_pymnt_inv   -0.324251
bc_open_to_buy    -0.080455
mort_acc          -0.080288
avg_cur_bal       -0.079429
tot_hi_cred_lim   -0.078612
tot_cur_bal       -0.071580
total_bc_limit    -0.071517
Name: target, dtype: float64


In [13]:
# List of columns that happen AFTER a loan is issued (Data Leakage)
leaky_cols = [
    'recoveries', 'collection_recovery_fee', 'total_rec_late_fee', 
    'out_prncp', 'out_prncp_inv', 'total_rec_prncp', 'total_pymnt', 
    'total_pymnt_inv', 'last_pymnt_amnt'
]

# Drop these columns from our dataframe (using errors='ignore' in case any were already dropped)
df = df.drop(columns=leaky_cols, errors='ignore')

print(f"Dropped {len(leaky_cols)} leaky columns.")
print(f"New Dataset Shape: {df.shape}")

Dropped 9 leaky columns.
New Dataset Shape: (1325535, 106)


In [14]:
# 1. Isolate all the text/object columns
categorical_cols = df.select_dtypes(include=['object']).columns

# 2. Count how many unique text values exist in each column
unique_counts = df[categorical_cols].nunique().sort_values(ascending=False)

print(f"Total Text/Categorical Columns: {len(categorical_cols)}\n")
print("Number of unique categories per column (Top 20 Highest):")
print(unique_counts.head(20))

print("\n" + "-"*40 + "\n")

print("Number of unique categories per column (Lowest):")
print(unique_counts.tail(10))

Total Text/Categorical Columns: 35

Number of unique categories per column (Top 20 Highest):
emp_title                    375011
desc                         121823
title                         61680
zip_code                        945
earliest_cr_line                738
sec_app_earliest_cr_line        572
last_credit_pull_d              140
issue_d                         139
last_pymnt_d                    135
settlement_date                  89
debt_settlement_flag_date        83
addr_state                       51
sub_grade                        35
hardship_end_date                28
payment_plan_start_date          27
hardship_start_date              27
purpose                          14
emp_length                       12
hardship_reason                  10
grade                             7
dtype: int64

----------------------------------------

Number of unique categories per column (Lowest):
next_pymnt_d            4
verification_status     3
term                    2
pymn

In [15]:
print("Starting Category Cleanup...")

# 1. Drop the Monsters, Redundancies, and Leaky Flags
cols_to_drop = [
    'emp_title', 'title', 'zip_code', 'earliest_cr_line', # Too complex / Date format
    'sub_grade',                                          # Redundant with 'grade'
    'hardship_flag', 'debt_settlement_flag', 'pymnt_plan' # Future Data / Leaky
]

df = df.drop(columns=cols_to_drop, errors='ignore')
print(f"Dropped {len(cols_to_drop)} messy/leaky text columns.")

# 2. Translate the remaining text into numbers (One-Hot Encoding)
# drop_first=True prevents the "Dummy Variable Trap" (statistical redundancy)
print("Converting remaining text to numbers...")
df = pd.get_dummies(df, drop_first=True)

# 3. Save this glorious, fully numeric, ML-ready dataset!
df.to_parquet('../data/loan_ml_ready.parquet', engine='pyarrow')
print("\n🎉 SUCCESS: Data is 100% numeric and saved!")
print(f"Final Machine-Learning-Ready Dataset Shape: {df.shape}")

Starting Category Cleanup...
Dropped 8 messy/leaky text columns.
Converting remaining text to numbers...


MemoryError: Unable to allocate 150. GiB for an array with shape (1325535, 121823) and data type bool

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.impute import SimpleImputer # <-- NEW IMPORT

print("1. Defining features (X) and target (y)...")
X = df.drop('target', axis=1)
y = df['target']

print("2. Splitting data into 80% Train and 20% Test...")
# stratify=y ensures our 78/21 imbalance is perfectly maintained in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("3. Handling Missing Values (Imputation)...")
# Fill any remaining blank cells with the median value of that specific column
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test) # Transform only to prevent data leakage!

print("4. Scaling the features (this normalizes large numbers like income)...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("5. Training the Baseline Logistic Regression Model...")
print("(This might take 60-90 seconds...)")
# class_weight='balanced' forces the model to pay equal attention to defaulters
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
log_reg.fit(X_train_scaled, y_train)

print("6. Making predictions on the unseen Test data...")
y_pred = log_reg.predict(X_test_scaled)

print("\n" + "="*40)
print("--- BASELINE MODEL REPORT CARD ---")
print("="*40)
print(classification_report(y_test, y_pred))

1. Defining features (X) and target (y)...
2. Splitting data into 80% Train and 20% Test...
3. Handling Missing Values (Imputation)...


ValueError: Cannot use median strategy with non-numeric data:
could not convert string to float: ' 36 months'

In [20]:
import xgboost as xgb
from sklearn.metrics import classification_report

print("1. Calculating the imbalance weight...")
# scale_pos_weight = total negative cases / total positive cases
weight = y_train.value_counts()[0] / y_train.value_counts()[1]
print(f"Weight for minority class: {weight:.2f}")

print("2. Initializing the XGBoost Classifier...")
xgb_model = xgb.XGBClassifier(
    n_estimators=100,       # Number of trees
    learning_rate=0.1,      # Step size for learning
    max_depth=5,            # How complex each tree can get
    scale_pos_weight=weight, # Handling the 78/21 imbalance
    random_state=42,
    n_jobs=-1               # Use all your laptop's CPU cores to make it fast!
)

print("3. Training the Champion Model (This might take 1-3 minutes)...")
xgb_model.fit(X_train_scaled, y_train)

print("4. Making predictions...")
y_pred_xgb = xgb_model.predict(X_test_scaled)

print("\n" + "="*40)
print("--- XGBOOST CHAMPION REPORT CARD ---")
print("="*40)
print(classification_report(y_test, y_pred_xgb))

1. Calculating the imbalance weight...
Weight for minority class: 3.67
2. Initializing the XGBoost Classifier...
3. Training the Champion Model (This might take 1-3 minutes)...


NameError: name 'X_train_scaled' is not defined

In [19]:
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
import matplotlib.pyplot as plt

# 1. Get the actual PROBABILITIES of default, not just the 0 or 1 guess
print("1. Calculating Probabilities of Default (PoD)...")
# [:, 1] grabs the probability of Class 1 (Default)
y_pred_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]

# 2. Calculate the ROC-AUC Score 
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC Score: {auc_score:.4f}")
print("(0.5 = Random, 0.7+ = Good, 0.8+ = Excellent)\n")

# 3. Calculate Precision and Recall across ALL possible thresholds
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
pr_auc = auc(recall, precision)

# 4. Plot the Precision-Recall Curve
plt.figure(figsize=(10, 6))
plt.plot(recall, precision, color='darkorange', lw=2, label=f'XGBoost (PR AUC = {pr_auc:.3f})')
plt.xlabel('Recall (Catching the actual Defaulters)')
plt.ylabel('Precision (Not rejecting Good Customers)')
plt.title('Precision-Recall Curve: The Business Trade-off')
plt.legend(loc="upper right")
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

1. Calculating Probabilities of Default (PoD)...


NameError: name 'xgb_model' is not defined

In [18]:
import joblib
import os

print("1. Ensuring the 'app' directory exists...")
os.makedirs('../app', exist_ok=True)

print("2. Saving the ML Pipeline components...")
# Save the imputer (to handle missing web form data)
joblib.dump(imputer, '../app/imputer.joblib')

# Save the scaler (to normalize the web form data)
joblib.dump(scaler, '../app/scaler.joblib')

# Save the Champion XGBoost Model
joblib.dump(xgb_model, '../app/xgb_model.joblib')

# Save the exact list of the 158 columns our model expects
# (This is critical so our web app formats the user's input correctly)
expected_columns = list(X.columns)
joblib.dump(expected_columns, '../app/expected_columns.joblib')

print("✅ SUCCESS: Model, Scaler, Imputer, and Columns saved to the 'app' folder!")

1. Ensuring the 'app' directory exists...
2. Saving the ML Pipeline components...


NameError: name 'scaler' is not defined

In [17]:
import joblib

# Calculate the median for all 158 columns
baseline_profile = X.median().to_dict()

# Save this "Average Joe" profile to the app folder
joblib.dump(baseline_profile, '../app/baseline_profile.joblib')
print("✅ SUCCESS: Baseline profile saved!")

TypeError: Cannot convert [[' 36 months' ' 60 months' ' 36 months' ... ' 60 months' ' 60 months'
  ' 60 months']
 ['D' 'C' 'A' ... 'F' 'C' 'E']
 ['5 years' '< 1 year' '10+ years' ... '10+ years' '< 1 year' '< 1 year']
 ...
 ['nan' 'nan' 'nan' ... 'nan' 'nan' 'nan']
 ['nan' 'nan' 'nan' ... 'nan' 'nan' 'nan']
 ['nan' 'nan' 'nan' ... 'nan' 'nan' 'nan']] to numeric

In [22]:
import pandas as pd
import numpy as np
import joblib

print("Loading final ML-ready data...")
# Skip the fragile Pickle and load the final Parquet file we made!
df = pd.read_parquet('../data/loan_ml_ready.parquet', engine='pyarrow')

print(f"Data loaded successfully! Shape: {df.shape}")

# 1. Re-define X (our features) and y (our target)
X = df.drop('target', axis=1)
y = df['target']

# 2. Calculate the median for all 158 columns
baseline_profile = X.median().to_dict()

# 3. Save this "Average Joe" profile to the app folder
joblib.dump(baseline_profile, '../app/baseline_profile.joblib')
print("✅ SUCCESS: Baseline profile saved!.")


Loading final ML-ready data...
Data loaded successfully! Shape: (1325535, 158)
✅ SUCCESS: Baseline profile saved!.
